# Generic Binary Guardrail Tuning

This notebook demonstrates the generic `BinaryGuardrailTuner` API. The example uses `PromptInjection`, but the same pattern works for any guardrail that classifies inputs into valid (`True`) or invalid (`False`).

You only provide CSV data with these columns:
- `text`: the text to scan
- `label`: `True` means valid, `False` means invalid

The scanner class owns its optimization space, and the tuner picks the default optimizer for you. You can optionally switch the optimizer by name.

If a scanner needs fixed non-optimized arguments, provide them through `SCANNER_FIXED_KWARGS`.

In [1]:
import csv
import json
from pathlib import Path

from dotenv import load_dotenv

from testsavant.guard import BinaryGuardrailTuner
from testsavant.guard.input_scanners import PromptInjection

load_dotenv()

True

In [2]:
TRAIN_CSV_PATH = Path('./sampled.csv')
TEST_CSV_PATH = None  # Optional: Path('/home/leo/project/ts/junk/v3-eval/test.csv')
TEXT_COLUMN = 'text'
LABEL_COLUMN = 'label'
SCANNER_CLS = PromptInjection
SCANNER_FIXED_KWARGS = {}  # Example: {'topics': ['finance'], 'mode': 'blacklist'}
OPTIMIZER_NAME = 'optuna'  # Optional: 'optuna'
MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None
RANDOM_STATE = 42
EPOCHS = 1  # Uses scanner defaults when None
BATCH_SIZE = 16  # Uses scanner defaults when None

In [3]:
scanner_defaults = SCANNER_CLS.get_optimization_defaults()
resolved_epochs = EPOCHS if EPOCHS is not None else int(scanner_defaults.get('epochs', 8))
resolved_batch_size = BATCH_SIZE if BATCH_SIZE is not None else int(scanner_defaults.get('batch_size', 32))
resolved_top_k = int(scanner_defaults.get('top_k', 10))

print({
    'scanner': SCANNER_CLS.__name__,
    'optimizer_name': OPTIMIZER_NAME,
    'epochs': resolved_epochs,
    'batch_size': resolved_batch_size,
    'top_k': resolved_top_k,
})

{'scanner': 'PromptInjection', 'optimizer_name': 'optuna', 'epochs': 1, 'batch_size': 16, 'top_k': 10}


In [4]:
def parse_valid_label(value):
    if isinstance(value, bool):
        return value

    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y', 'valid'}:
        return True
    if normalized in {'false', '0', 'no', 'n', 'invalid'}:
        return False
    raise ValueError(f'Unsupported label value: {value!r}')


def load_csv_dataset(path: Path, text_column: str, label_column: str, max_rows=None):
    rows = []
    with path.open('r', encoding='utf-8', newline='') as csv_file:
        reader = csv.DictReader(csv_file)
        for raw_row in reader:
            text = raw_row[text_column]
            label = parse_valid_label(raw_row[label_column])
            rows.append((text, label))
            if max_rows is not None and len(rows) >= max_rows:
                break
    return rows


train_rows = load_csv_dataset(TRAIN_CSV_PATH, TEXT_COLUMN, LABEL_COLUMN, max_rows=MAX_TRAIN_ROWS)
# split to train to test if no test csv is provided
test_rows = None
if TEST_CSV_PATH is not None:
    test_rows = load_csv_dataset(TEST_CSV_PATH, TEXT_COLUMN, LABEL_COLUMN, max_rows=MAX_TEST_ROWS)
else:
    # split to train to test if no test csv is provided, 
    split_index = int(0.7 * len(train_rows))
    test_rows = train_rows[split_index:]
    train_rows = train_rows[:split_index]

print({
    'train_rows': len(train_rows),
    'test_rows': len(test_rows) if test_rows is not None else 0,
})

{'train_rows': 70, 'test_rows': 30}


In [5]:
tuner = BinaryGuardrailTuner.from_input_scanner_class(
    scanner_cls=SCANNER_CLS,
    fixed_scanner_kwargs=SCANNER_FIXED_KWARGS,
    optimizer_name=OPTIMIZER_NAME,
    top_k=resolved_top_k,
    random_state=RANDOM_STATE,
)

tuner

In [6]:
def on_step(payload):
    if not payload['top_results']:
        return
    best_result = payload['top_results'][0]
    print(
        f"step={payload['step']}/{payload['total_steps']} "
        f"best_mean_score={best_result['mean_score']:.4f} "
        f"best_objective_score={best_result['objective_score']:.4f} "
        f"config={best_result['config']}"
    )


fit_kwargs = {
    'train_x': [text for text, _ in train_rows],
    'train_y': [label for _, label in train_rows],
    'epochs': resolved_epochs,
    'batch_size': resolved_batch_size,
    'on_step': on_step,
}

if test_rows is not None:
    fit_kwargs['test_x'] = [text for text, _ in test_rows]
    fit_kwargs['test_y'] = [label for _, label in test_rows]

fit_result = tuner.fit(**fit_kwargs)

fit_result.best_config

[I 2026-07-10 18:47:24,322] A new study created in memory with name: no-name-44cfe70c-779f-4059-aeac-8533c4ed287c
[I 2026-07-10 18:47:25,180] Trial 0 finished with value: 0.7576145637162587 and parameters: {'threshold': 0.275, 'chunk_size': 275, 'overlap': 20}. Best is trial 0 with value: 0.7576145637162587.


step=1/5 best_mean_score=0.7458 best_objective_score=0.7576 config={'threshold': 0.275, 'chunk_size': 275, 'overlap': 20}


[I 2026-07-10 18:47:25,876] Trial 1 finished with value: 0.8667724867724867 and parameters: {'threshold': 0.25, 'chunk_size': 350, 'overlap': 30}. Best is trial 1 with value: 0.8667724867724867.


step=2/5 best_mean_score=0.8571 best_objective_score=0.8668 config={'threshold': 0.25, 'chunk_size': 350, 'overlap': 30}


[I 2026-07-10 18:47:26,648] Trial 2 finished with value: 0.9211111111111111 and parameters: {'threshold': 0.9, 'chunk_size': 200, 'overlap': 50}. Best is trial 2 with value: 0.9211111111111111.


...

[I 2026-07-10 18:47:27,429] Trial 3 finished with value: 0.0059259259259259265 and parameters: {'threshold': 0.025, 'chunk_size': 250, 'overlap': 50}. Best is trial 2 with value: 0.9211111111111111.


step=4/5 best_mean_score=0.9167 best_objective_score=0.9211 config={'threshold': 0.9, 'chunk_size': 200, 'overlap': 50}


[I 2026-07-10 18:47:28,242] Trial 4 finished with value: 0.7670899470899472 and parameters: {'threshold': 0.625, 'chunk_size': 125, 'overlap': 30}. Best is trial 2 with value: 0.9211111111111111.


step=5/5 best_mean_score=0.9167 best_objective_score=0.9211 config={'threshold': 0.9, 'chunk_size': 200, 'overlap': 50}


{'threshold': 0.625, 'chunk_size': 125, 'overlap': 30}

In [7]:
report = {
    'report_split_name': fit_result.report_split_name,
    'best_config': fit_result.best_config,
    'best_train_metrics': fit_result.best_train_metrics.to_dict(),
    'best_report_metrics': fit_result.best_report_metrics.to_dict(),
    'best_test_metrics': fit_result.best_test_metrics.to_dict() if fit_result.best_test_metrics is not None else None,
}

print(json.dumps(report, indent=2))

{
  "report_split_name": "test",
  "best_config": {
    "threshold": 0.625,
    "chunk_size": 125,
    "overlap": 30
  },
  "best_train_metrics": {
    "total_count": 70,
    "valid_count": 17,
    "invalid_count": 53,
    "true_positive": 39,
    "false_positive": 2,
    "true_negative": 15,
    "false_negative": 14,
    "recall": 0.7358490566037735,
    "specificity": 0.8823529411764706,
    "precision": 0.9512195121951219,
    "f1_score": 0.8297872340425531,
    "false_positive_rate": 0.11764705882352941,
    "false_negative_rate": 0.2641509433962264,
    "accuracy": 0.7714285714285715,
    "effectiveness_score": 0.802469135802469,
    "selection_score": 0.802469135802469,
    "predicted_valid_count": 29,
    "predicted_invalid_count": 41
  },
  "best_report_metrics": {
    "total_count": 30,
    "valid_count": 7,
    "invalid_count": 23,
    "true_positive": 14,
    "false_positive": 1,
    "true_negative": 6,
    "false_negative": 9,
    "recall": 0.6086956521739131,
    "specific